# Model Versioning with MLflow

## 📚 Learning Objectives

By completing this notebook, you will:
- Version models and configs
- Roll back and A/B test deployments

## 🔗 Prerequisites

- ✅ Basic Python
- ✅ Basic NumPy/Pandas (when applicable)

---

## Official Structure Reference

This notebook supports **Course 11, Unit 2** requirements from `DETAILED_UNIT_DESCRIPTIONS.md`.

---


# Model Versioning with MLflow
## AIAT 125 - Model Deployment

## Learning Objectives

- Understand model versioning
- Use MLflow for model registry
- Track model versions
- Manage model lifecycle

## Real-World Context

Model management, tracking, and version control in production.

**Industry Impact**: Essential for production ML systems.

## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [ ]:
%pip install mlflow -q
import mlflow
import mlflow.sklearn
print('✅ Setup complete!')


## Part 1: MLflow Setup


In [ ]:
# Set MLflow tracking URI (use local file to avoid connection timeout)
mlflow.set_tracking_uri('file:./mlruns')  # Local file instead of server

# Start experiment
mlflow.set_experiment('model_versioning_demo')
print('✅ MLflow setup complete')

## Part 2: Logging Models


In [ ]:
# Example: Log model with MLflow
with mlflow.start_run():
    # Log parameters
    mlflow.log_param('n_estimators', 100)
    mlflow.log_param('max_depth', 10)
    
    # Log metrics
    mlflow.log_metric('accuracy', 0.95)
    mlflow.log_metric('f1_score', 0.93)
 
 # Log model
 # mlflow.sklearn.log_model(model, 'model')
print('✅ Model logged to MLflow')
print('\nReal-world: Track all model versions and experiments')

## Part 3: Model Registry


In [ ]:
print('📝 Model Registry Concept:')
print('\n1. Register models with versions')
print('2. Tag models (production, staging, development)')
print('3. Track model lineage')
print('4. Manage model lifecycle')
print('\n✅ Model registry understood!')
print('\nReal-world: Production model management')

## Real-World Applications

- **Production**: Track deployed models
- **Experiments**: Compare model versions
- **Compliance**: Audit model changes
- **Collaboration**: Share models across teams

---

**End of Notebook**

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

versions = ['v1.0\n(2024-01)', 'v1.1\n(2024-03)', 'v2.0\n(2024-06)']
x_pos = [1, 3, 5]
colors = ['#3498db', '#27ae60', '#e74c3c']
descriptions = ['Initial release\nAcc: 85%', 'Bug fixes\nAcc: 87%', 'New arch\nAcc: 93%']

fig, ax = plt.subplots(figsize=(10, 4))
ax.set_xlim(0, 6.5)
ax.set_ylim(-1, 2)
ax.axis('off')
ax.set_title('Model Version Timeline', fontsize=14, fontweight='bold', pad=12)

# Timeline line
ax.plot([0.5, 6], [0.5, 0.5], color='gray', linewidth=3, zorder=1)

for i, (x, version, color, desc) in enumerate(zip(x_pos, versions, colors, descriptions)):
    # Circle marker
    circle = plt.Circle((x, 0.5), 0.25, color=color, ec='black', linewidth=1.5, zorder=3)
    ax.add_patch(circle)
    ax.text(x, 0.5, str(i+1), ha='center', va='center', fontsize=12,
            fontweight='bold', color='white', zorder=4)
    # Version label above
    ax.text(x, 1.15, version, ha='center', va='bottom', fontsize=10, fontweight='bold')
    # Description below
    ax.text(x, -0.2, desc, ha='center', va='top', fontsize=8.5, color='#555')

fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


## 🌍 Real-World Worked Example — Deploy a Trained Model as a REST API

**Industry context:**
- Spotify's recommendation model is served via a FastAPI microservice handling 400M users
- Instagram's image moderation runs as a containerised PyTorch model behind a REST endpoint
- Every ML feature in a modern app goes through a model serving layer like this

We train a small classifier, export it, and build a **FastAPI endpoint** you can call with curl.

In [ ]:
# ── Part 1: Train and save a model ────────────────────────────────────────
import torch, torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

iris = load_iris()
X = StandardScaler().fit_transform(iris.data.astype(np.float32))
y = iris.target
X_tr,X_te,y_tr,y_te = train_test_split(X, y, test_size=0.2, random_state=42)

model = nn.Sequential(nn.Linear(4,32), nn.ReLU(), nn.Linear(32,3))
opt   = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()
Xt = torch.tensor(X_tr); Yt = torch.tensor(y_tr, dtype=torch.long)

for _ in range(200):
    loss = loss_fn(model(Xt), Yt)
    opt.zero_grad(); loss.backward(); opt.step()

torch.save(model.state_dict(), '/tmp/iris_model.pt')
print("Model saved to /tmp/iris_model.pt")

# Verify
model.eval()
with torch.no_grad():
    acc = (model(torch.tensor(X_te)).argmax(1)==torch.tensor(y_te)).float().mean()
print(f"Test accuracy: {acc:.2%}")

# ── Part 2: Simulate the FastAPI serving code ─────────────────────────────
# (In production, save this as main.py and run: uvicorn main:app --reload)
fastapi_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import torch, torch.nn as nn
import numpy as np

app = FastAPI(title="Iris Classifier API")

# Load model at startup
model = nn.Sequential(nn.Linear(4,32), nn.ReLU(), nn.Linear(32,3))
model.load_state_dict(torch.load("/tmp/iris_model.pt"))
model.eval()
CLASSES = ["setosa", "versicolor", "virginica"]

class IrisRequest(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.post("/predict")
def predict(req: IrisRequest):
    features = torch.tensor([[req.sepal_length, req.sepal_width,
                               req.petal_length, req.petal_width]])
    with torch.no_grad():
        logits = model(features)
        probs  = torch.softmax(logits, dim=1)[0]
        label  = CLASSES[probs.argmax().item()]
    return {"prediction": label, "confidence": round(probs.max().item(), 3)}

@app.get("/health")
def health(): return {"status": "ok"}

# Run with: uvicorn main:app --host 0.0.0.0 --port 8000
# Test with: curl -X POST http://localhost:8000/predict -H "Content-Type: application/json" \
#            -d '{"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}'
'''
print("\n── FastAPI serving code (save as main.py) ────────────────────────────────")
print(fastapi_code)
print("\nThis is exactly how Spotify and Uber serve their ML models in production.")

## 📚 References & Further Reading

**Frameworks:**
- [FastAPI Documentation](https://fastapi.tiangolo.com/) — Modern Python API framework
- [ONNX Runtime](https://onnxruntime.ai/) — Cross-platform inference
- [BentoML](https://github.com/bentoml/BentoML) — ML model serving framework

**Cloud Services:**
- [AWS SageMaker Inference](https://docs.aws.amazon.com/sagemaker/latest/dg/deploy-model.html)
- [Google Cloud Vertex AI](https://cloud.google.com/vertex-ai/docs/predictions/overview)

**State-of-the-Art:** Uber, Airbnb, and Spotify deploy hundreds of ML models using microservices with FastAPI/gRPC.

## 📝 Summary

You learned **model versioning and experiment tracking** — recording hyperparameters, metrics, and artifacts. MLflow, Weights & Biases, and Neptune are the standard tools for this. Without tracking, reproducing results or rolling back to a better model becomes impossible.